In [8]:
import os

import numpy as np
import scipy.stats as stats
import pandas as pd
from util import stat_m_e

nsubj = 8
npos = 16
nroi = 7
num_nf = [[0, 4], [5, 9]]
ntask = 2
nf = ['Near', 'Far']
roi_labels = ['V1', 'V2', 'V3', 'hV4', 'IOG', 'pFus', 'mFus']

data_path = "../data/"
out_xlsx = 'sourcedata.xlsx'


def _write_sheet(out_xlsx: str, sheet: str, df: pd.DataFrame) -> str:
    mode = 'a' if os.path.exists(out_xlsx) else 'w'
    writer_kwargs = dict(engine='openpyxl', mode=mode)
    if mode == 'a':
        writer_kwargs['if_sheet_exists'] = 'replace'
    with pd.ExcelWriter(out_xlsx, **writer_kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet)
    return out_xlsx


def export_fig2_like(out_xlsx, sheet, nf_subj, *, roi_labels=roi_labels):
    """Fig2B/C: sample×roi×task×nearfar -> one sheet (task vertical, distance horizontal)."""

    assert nf_subj.ndim == 4
    assert nf_subj.shape[1] == len(roi_labels)
    assert nf_subj.shape[2] == 2, f"期望 task=2，但得到 {nf_subj.shape[2]}"
    assert nf_subj.shape[3] == 2, f"期望 nf=2，但得到 {nf_subj.shape[3]}"

    section_titles = {0: 'digit_task', 1: 'face_task'}
    nf_labels = {0: 'Near', 1: 'Far'}

    def _one_task_block(task_i: int) -> pd.DataFrame:
        parts = []
        for nf_i in range(nf_subj.shape[3]):
            dist = nf_labels.get(nf_i, f'nf_{nf_i}')
            df_part = pd.DataFrame(nf_subj[:, :, task_i, nf_i], columns=roi_labels)
            df_part.columns = pd.MultiIndex.from_product([[dist], roi_labels])
            parts.append(df_part)

        df_raw = pd.concat(parts, axis=1)
        df_raw.index = np.arange(1, df_raw.shape[0] + 1)
        df_raw.index.name = 'sample'

        df_stats = pd.DataFrame(index=['mean', 'sem'], columns=df_raw.columns, dtype=float)
        for nf_i in range(nf_subj.shape[3]):
            dist = nf_labels.get(nf_i, f'nf_{nf_i}')
            m, e, _ = stat_m_e(nf_subj[:, :, task_i, nf_i], 'mean', 'sem')
            df_stats.loc['mean', (dist, slice(None))] = m
            df_stats.loc['sem', (dist, slice(None))] = e

        df_out = pd.concat([df_raw, df_stats], axis=0)
        df_title = pd.DataFrame(index=[section_titles.get(task_i, f'task_{task_i}')], columns=df_out.columns)
        df_blank = pd.DataFrame(index=[''], columns=df_out.columns)
        return pd.concat([df_title, df_out, df_blank], axis=0)

    df_sheet = pd.concat([_one_task_block(0), _one_task_block(1)], axis=0)
    return _write_sheet(out_xlsx, sheet, df_sheet)


def _bootstrap_ci_median(data, *, n_boot=10000, ci=95, rng=None):
    """Median CI from bootstrap resampling over the first axis."""

    if rng is None:
        rng = np.random.default_rng(0)

    data = np.asarray(data, dtype=float)
    alpha = (100 - ci) / 2
    ci_low = np.full(data.shape[1], np.nan)
    ci_high = np.full(data.shape[1], np.nan)
    for col_i in range(data.shape[1]):
        x = data[:, col_i]
        x = x[~np.isnan(x)]
        if x.size == 0:
            continue
        boot_idx = rng.integers(0, x.size, size=(n_boot, x.size))
        boot_median = np.median(x[boot_idx], axis=1)
        ci_low[col_i], ci_high[col_i] = np.percentile(boot_median, [alpha, 100 - alpha])
    return ci_low, ci_high


def _distribution_stats(data, *, roi_labels=roi_labels, n_boot=10000, ci=95, rng=None):
    median = np.nanmedian(data, axis=0)
    percentile_25, percentile_75 = np.nanpercentile(data, [25, 75], axis=0)
    iqr_1p5 = 1.5 * (percentile_75 - percentile_25)
    ci_low, ci_high = _bootstrap_ci_median(data, n_boot=n_boot, ci=ci, rng=rng)
    return pd.DataFrame(
        [median, ci_low, ci_high, percentile_25, percentile_75, iqr_1p5],
        index=['median', 'ci95_low_bootstrap_median', 'ci95_high_bootstrap_median', 'percentile_25', 'percentile_75', '1.5x_iqr'],
        columns=roi_labels,
    )


def export_fig3_like(out_xlsx, sheet, v_subj, *, roi_labels=roi_labels, include_raw=True, extra_distribution_stats=False, n_boot=10000, random_seed=0):
    """Fig3/4: sample×roi×task -> one sheet (task vertical)."""

    assert v_subj.ndim == 3
    assert v_subj.shape[1] == len(roi_labels)
    assert v_subj.shape[2] == 2, f"期望 task=2，但得到 {v_subj.shape[2]}"

    section_titles = {0: 'digit_task', 1: 'face_task'}
    rng = np.random.default_rng(random_seed)

    def _one_task_block(task_i: int) -> pd.DataFrame:
        df_raw = pd.DataFrame(v_subj[:, :, task_i], columns=roi_labels)
        df_raw.index = np.arange(1, df_raw.shape[0] + 1)
        df_raw.index.name = 'sample'

        m, e, _ = stat_m_e(v_subj[:, :, task_i], 'mean', 'sem')
        df_stats = pd.DataFrame([m, e], index=['mean', 'sem'], columns=roi_labels)
        if extra_distribution_stats:
            df_stats = pd.concat(
                [df_stats, _distribution_stats(v_subj[:, :, task_i], roi_labels=roi_labels, n_boot=n_boot, rng=rng)],
                axis=0,
            )

        df_out = pd.concat([df_raw, df_stats], axis=0) if include_raw else df_stats
        df_title = pd.DataFrame(index=[section_titles.get(task_i, f'task_{task_i}')], columns=df_out.columns)
        df_blank = pd.DataFrame(index=[''], columns=df_out.columns)
        return pd.concat([df_title, df_out, df_blank], axis=0)

    df_sheet = pd.concat([_one_task_block(0), _one_task_block(1)], axis=0)
    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_fig2_no_task(out_xlsx, sheet, nf_subj, *, roi_labels=roi_labels):
    """No-task case: sample×roi×nearfar -> one sheet (Near/Far left/right)."""

    assert nf_subj.ndim == 3
    assert nf_subj.shape[1] == len(roi_labels)
    assert nf_subj.shape[2] == 2, f"期望 nf=2，但得到 {nf_subj.shape[2]}"

    nf_labels = {0: 'Near', 1: 'Far'}

    parts = []
    for nf_i in range(nf_subj.shape[2]):
        dist = nf_labels.get(nf_i, f'nf_{nf_i}')
        df_part = pd.DataFrame(nf_subj[:, :, nf_i], columns=roi_labels)
        df_part.columns = pd.MultiIndex.from_product([[dist], roi_labels])
        parts.append(df_part)

    df_raw = pd.concat(parts, axis=1)
    df_raw.index = np.arange(1, df_raw.shape[0] + 1)
    df_raw.index.name = 'sample'

    df_stats = pd.DataFrame(index=['mean', 'sem'], columns=df_raw.columns, dtype=float)
    for nf_i in range(nf_subj.shape[2]):
        dist = nf_labels.get(nf_i, f'nf_{nf_i}')
        m, e, _ = stat_m_e(nf_subj[:, :, nf_i], 'mean', 'sem')
        df_stats.loc['mean', (dist, slice(None))] = m
        df_stats.loc['sem', (dist, slice(None))] = e

    df_sheet = pd.concat([df_raw, df_stats], axis=0)
    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_voxel_summary_task_nf(out_xlsx, sheet, x, *, roi_labels=roi_labels):
    """Voxel summary (no raw data).

    x: (nvoxel, nsample, nroi, ntask, nnf)
    输出：上下 task；每个 task 里先 mean 后 sem；列为 Near/Far × ROI；行是 voxel_1..voxel_k
    """

    assert x.ndim == 5
    nvoxel, nsample, nroi_, ntask_, nnf_ = x.shape
    assert nroi_ == len(roi_labels)
    assert ntask_ == 2
    assert nnf_ == 2

    section_titles = {0: 'digit_task', 1: 'face_task'}
    nf_labels = {0: 'Near', 1: 'Far'}

    def _stats_block(task_i: int, stat: str) -> pd.DataFrame:
        if stat == 'mean':
            vals = np.nanmean(x[:, :, :, task_i, :], axis=1)  # voxel x roi x nf
        elif stat == 'sem':
            vals = stats.sem(x[:, :, :, task_i, :], axis=1, nan_policy='omit')
        else:
            raise ValueError(stat)

        parts = []
        for nf_i in range(nnf_):
            dist = nf_labels.get(nf_i, f'nf_{nf_i}')
            df_part = pd.DataFrame(vals[:, :, nf_i], columns=roi_labels)
            df_part.columns = pd.MultiIndex.from_product([[dist], roi_labels])
            parts.append(df_part)

        df = pd.concat(parts, axis=1)
        df.index = [f'voxel_{i+1}' for i in range(df.shape[0])]
        df.index.name = 'voxel'
        return df

    columns = pd.MultiIndex.from_product([['Near', 'Far'], roi_labels])
    blank = pd.DataFrame(index=[''], columns=columns)

    blocks = []
    for task_i in range(ntask_):
        blocks.append(pd.DataFrame(index=[section_titles.get(task_i, f'task_{task_i}')], columns=columns))
        blocks.append(_stats_block(task_i, 'mean'))
        blocks.append(blank)
        blocks.append(_stats_block(task_i, 'sem'))
        blocks.append(blank)

    df_sheet = pd.concat(blocks, axis=0)
    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_voxel_summary_no_task_nf(out_xlsx, sheet, x, *, roi_labels=roi_labels):
    """Voxel summary, no task.

    x: (nvoxel, nsample, nroi, nnf)
    输出：先 mean 再 sem；列为 Near/Far × ROI；行是 voxel_1..voxel_k
    """

    assert x.ndim == 4
    nvoxel, nsample, nroi_, nnf_ = x.shape
    assert nroi_ == len(roi_labels)
    assert nnf_ == 2

    nf_labels = {0: 'Near', 1: 'Far'}

    def _stats_block(stat: str) -> pd.DataFrame:
        if stat == 'mean':
            vals = np.nanmean(x, axis=1)  # voxel x roi x nf
        elif stat == 'sem':
            vals = stats.sem(x, axis=1, nan_policy='omit')
        else:
            raise ValueError(stat)

        parts = []
        for nf_i in range(nnf_):
            dist = nf_labels.get(nf_i, f'nf_{nf_i}')
            df_part = pd.DataFrame(vals[:, :, nf_i], columns=roi_labels)
            df_part.columns = pd.MultiIndex.from_product([[dist], roi_labels])
            parts.append(df_part)

        df = pd.concat(parts, axis=1)
        df.index = [f'voxel_{i+1}' for i in range(df.shape[0])]
        df.index.name = 'voxel'
        return df

    columns = pd.MultiIndex.from_product([['Near', 'Far'], roi_labels])
    blank = pd.DataFrame(index=[''], columns=columns)

    df_sheet = pd.concat(
        [
            pd.DataFrame(index=['mean'], columns=columns),
            _stats_block('mean'),
            blank,
            pd.DataFrame(index=['sem'], columns=columns),
            _stats_block('sem'),
        ],
        axis=0,
    )

    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_pc_summary(out_xlsx, sheet, x, *, roi_labels=roi_labels):
    """PC summary only, no raw data.

    x: (npc, nsample, nroi)
    输出：列为 ROI；行为 mean_PC1..mean_PCk 和 sem_PC1..sem_PCk
    """

    assert x.ndim == 3
    npc, nsample, nroi_ = x.shape
    assert nroi_ == len(roi_labels)

    mean_vals = np.nanmean(x, axis=1)
    sem_vals = stats.sem(x, axis=1, nan_policy='omit')

    df_mean = pd.DataFrame(mean_vals, index=[f'mean_PC{i+1}' for i in range(npc)], columns=roi_labels)
    df_sem = pd.DataFrame(sem_vals, index=[f'sem_PC{i+1}' for i in range(npc)], columns=roi_labels)
    df_sheet = pd.concat([df_mean, df_sem], axis=0)
    df_sheet.index.name = 'stat_pc'

    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_pc_summary_by_task(out_xlsx, sheet, x, roi_label):
    """PC summary for one ROI, split by task, no raw data.

    x: (npc, nsample, ntask)
    输出：上下 task；每个 task 里 mean_PC1..mean_PCk 和 sem_PC1..sem_PCk；列为单个 ROI
    """

    assert x.ndim == 3
    npc, nsample, ntask_ = x.shape
    assert ntask_ == 2

    section_titles = {0: 'digit_task', 1: 'face_task'}
    columns = [roi_label]
    blank = pd.DataFrame(index=[''], columns=columns)

    blocks = []
    for task_i in range(ntask_):
        mean_vals = np.nanmean(x[:, :, task_i], axis=1)
        sem_vals = stats.sem(x[:, :, task_i], axis=1, nan_policy='omit')
        df_mean = pd.DataFrame(mean_vals, index=[f'mean_PC{i+1}' for i in range(npc)], columns=columns)
        df_sem = pd.DataFrame(sem_vals, index=[f'sem_PC{i+1}' for i in range(npc)], columns=columns)

        blocks.append(pd.DataFrame(index=[section_titles.get(task_i, f'task_{task_i}')], columns=columns))
        blocks.append(df_mean)
        blocks.append(df_sem)
        blocks.append(blank)

    df_sheet = pd.concat(blocks, axis=0)
    df_sheet.index.name = 'stat_pc'
    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_roi_type_samples(out_xlsx, sheet, x, type_labels, *, roi_labels=roi_labels):
    """Export samples by type for each ROI.

    x: (nsample, nroi, ntype)
    输出：每个 ROI 一个区块；行为 type；列为 sample_1..sample_n、mean、sem
    """

    assert x.ndim == 3
    nsample, nroi_, ntype = x.shape
    assert nroi_ == len(roi_labels)
    assert ntype == len(type_labels)

    columns = [f'sample_{i+1}' for i in range(nsample)] + ['mean', 'sem']
    blank = pd.DataFrame(index=[''], columns=columns)

    blocks = []
    for roi_i, roi_label in enumerate(roi_labels):
        roi_data = x[:, roi_i, :].T  # type x sample
        df = pd.DataFrame(roi_data, index=type_labels, columns=columns[:nsample])
        df['mean'] = np.nanmean(roi_data, axis=1)
        df['sem'] = stats.sem(roi_data, axis=1, nan_policy='omit')

        blocks.append(pd.DataFrame(index=[roi_label], columns=columns))
        blocks.append(df)
        blocks.append(blank)

    df_sheet = pd.concat(blocks, axis=0)
    df_sheet.index.name = 'roi_type'
    return _write_sheet(out_xlsx, sheet, df_sheet)


def export_fig7_summary(out_xlsx, sheet, x, roi_sel, metric_labels, *, roi_labels=roi_labels):
    """Fig7 summary, no raw data.

    x: (nsample, npc, ntask, nselected_roi, nmetric)
    输出：每个 ROI 一个区块；行为 task_metric_mean/sem；列为 PC_1..PC_n
    """

    assert x.ndim == 5
    nsample, npc, ntask_, nroi_sel, nmetric = x.shape
    assert ntask_ == 2
    assert nroi_sel == len(roi_sel)
    assert nmetric == len(metric_labels)

    section_titles = {0: 'digit_task', 1: 'face_task'}
    columns = [f'PC_{i+1}' for i in range(npc)]
    blank = pd.DataFrame(index=[''], columns=columns)

    blocks = []
    for j, roi_i in enumerate(roi_sel):
        blocks.append(pd.DataFrame(index=[roi_labels[roi_i]], columns=columns))
        rows = []
        row_index = []
        for task_i in range(ntask_):
            for metric_i, metric_label in enumerate(metric_labels):
                vals = x[:, :, task_i, j, metric_i]
                rows.append(np.nanmean(vals, axis=0))
                row_index.append(f'{section_titles[task_i]}_{metric_label}_mean')
                rows.append(stats.sem(vals, axis=0, nan_policy='omit'))
                row_index.append(f'{section_titles[task_i]}_{metric_label}_sem')

        blocks.append(pd.DataFrame(rows, index=row_index, columns=columns))
        blocks.append(blank)

    df_sheet = pd.concat(blocks, axis=0)
    df_sheet.index.name = 'roi_task_metric_stat'
    return _write_sheet(out_xlsx, sheet, df_sheet)


#### fig 2

In [2]:
# Fig2B
acc_d_subj = np.load(data_path + 'svmacc.npz', allow_pickle=True)['acc_d_subj']  # d x subj x roi x task

acc_nf_subj = np.full((4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for i in range(len(num_nf)):
            data = acc_d_subj[num_nf[i][0] : num_nf[i][1], :, roi_i, task_i]
            acc_nf_subj[:, roi_i, task_i, i] = data.reshape(-1)

# swap task axis to match figure convention
acc_nf_subj[:, :, [0, 1], :] = acc_nf_subj[:, :, [1, 0], :]

export_fig2_like(out_xlsx, sheet='Fig2b', nf_subj=acc_nf_subj)


# Fig2C
LFI_bc_d_subj = np.load(data_path + 'LFI_bc.npz', allow_pickle=True)['LFI_bc_d_subj']  # d x subj x roi x task

LFI_bc_nf_subj = np.full((4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for i in range(len(num_nf)):
            data = LFI_bc_d_subj[num_nf[i][0] : num_nf[i][1], :, roi_i, task_i]
            LFI_bc_nf_subj[:, roi_i, task_i, i] = data.reshape(-1)

# swap task axis to match figure convention
LFI_bc_nf_subj[:, :, [0, 1], :] = LFI_bc_nf_subj[:, :, [1, 0], :]

export_fig2_like(out_xlsx, sheet='Fig2c', nf_subj=LFI_bc_nf_subj)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

'sourcedata.xlsx'

#### fig 3

In [9]:
# Fig3A/3B/3C
vRF_allv = np.load(data_path + 'vRF.npz', allow_pickle=True)['vRF_allv']

vRFv = vRF_allv.reshape(
    vRF_allv.shape[0],
    vRF_allv.shape[1],
    vRF_allv.shape[2] * vRF_allv.shape[3],
    vRF_allv.shape[4],
)
vRFv = np.transpose(vRFv, axes=(2, 1, 0, 3))  # sample x roi x task x property

# swap task axis to match figure convention (digit/face)
vRFv[:, :, [0, 1], :] = vRFv[:, :, [1, 0], :]

# indices for eccentricity, size, gain
prop = {
    'Fig3a': (3, 'eccentricity'),
    'Fig3b': (5, 'size'),
    'Fig3c': (7, 'gain'),
}

for sheet, (pi, _) in prop.items():
    export_fig3_like(
        out_xlsx,
        sheet=sheet,
        v_subj=vRFv[:, :, :, pi],
        include_raw=False,
        extra_distribution_stats=True,
        n_boot=10000,
        random_seed=0,
    )

out_xlsx

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

'sourcedata.xlsx'

#### fig 4

In [10]:
# Fig4A/4B/4C
rvv_all = np.load(data_path + 'responsevariance.npz', allow_pickle=True)['rvv_all']
rvv = np.nanmedian(rvv_all, axis=1)
rvv = rvv.reshape(rvv.shape[0] * rvv.shape[1], rvv.shape[2], rvv.shape[3])
rvv[:, :, [0, 1]] = rvv[:, :, [1, 0]]  # 800 sample x 7 roi x 2 task

FF2v_all = np.load(data_path + 'FF.npz', allow_pickle=True)['FFv_all']
FF2v = np.nanmedian(FF2v_all, axis=1)
FF2v = FF2v.reshape(FF2v.shape[0] * FF2v.shape[1], FF2v.shape[2], FF2v.shape[3])
FF2v[:, :, [0, 1]] = FF2v[:, :, [1, 0]]  # 800 sample x 7 roi x 2 task

corr_all = np.load(data_path + 'corr.npz', allow_pickle=True)['corr_all']  # 128 sample x 7 roi x 2 task
corr_all[:, :, [0, 1]] = corr_all[:, :, [1, 0]]

export_fig3_like(out_xlsx, sheet='Fig4a', v_subj=rvv, include_raw=False, extra_distribution_stats=True, n_boot=10000, random_seed=0)
export_fig3_like(out_xlsx, sheet='Fig4b', v_subj=FF2v, include_raw=False, extra_distribution_stats=True, n_boot=10000, random_seed=0)
export_fig3_like(out_xlsx, sheet='Fig4c', v_subj=corr_all)

out_xlsx

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14020\123395228.py:128: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

'sourcedata.xlsx'

#### fig 5

In [5]:
# Fig5b/5d

dfnorm_d_subj = np.load(data_path + 'dfnorm.npz', allow_pickle=True)['dfnorm_d_subj']  # d x subj x roi x task
dfnorm_nf_subj = np.full((4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for i in range(len(num_nf)):
            data = dfnorm_d_subj[num_nf[i][0] : num_nf[i][1], :, roi_i, task_i]
            dfnorm_nf_subj[:, roi_i, task_i, i] = data.reshape(-1)
dfnorm_nf_subj[:, :, [0, 1], :] = dfnorm_nf_subj[:, :, [1, 0], :]

v_d_subj = np.load(data_path + 'meanvariance.npz', allow_pickle=True)['v_d_subj']  # d x subj x roi x task
v_nf_subj = np.full((4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for i in range(len(num_nf)):
            data = v_d_subj[num_nf[i][0] : num_nf[i][1], :, roi_i, task_i]
            v_nf_subj[:, roi_i, task_i, i] = data.reshape(-1)
v_nf_subj[:, :, [0, 1], :] = v_nf_subj[:, :, [1, 0], :]

export_fig2_like(out_xlsx, sheet='Fig5b', nf_subj=dfnorm_nf_subj)
export_fig2_like(out_xlsx, sheet='Fig5d', nf_subj=v_nf_subj)

out_xlsx

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

'sourcedata.xlsx'

In [6]:
# Fig5f (no task)
srangle_d_subj = np.load(data_path + 'srangle.npz', allow_pickle=True)['srangle_d_subj']  # d x subj x roi

srangle_nf_subj = np.full((4 * nsubj, nroi, len(nf)), np.nan)  # 32 x roi x nf
for roi_i in range(nroi):
    for i in range(len(num_nf)):
        data = srangle_d_subj[num_nf[i][0] : num_nf[i][1], :, roi_i]
        srangle_nf_subj[:, roi_i, i] = data.reshape(-1)

export_fig2_no_task(out_xlsx, sheet='Fig5f', nf_subj=srangle_nf_subj)

out_xlsx

'sourcedata.xlsx'

In [7]:
# Fig5f (no task)
srangle_d_subj_ctl = np.load(data_path + 'srangle_ctl.npz', allow_pickle=True)['srangle_ctl']  # d x subj x roi

for task_i in range(2):
    data_all = srangle_d_subj_ctl[task_i,:]
    srangle_nf_subj = np.full((4 * nsubj, nroi, len(nf)), np.nan)  # 32 x roi x nf
    for roi_i in range(nroi):
        for i in range(len(num_nf)):
            data = data_all[num_nf[i][0] : num_nf[i][1], :, roi_i]
            srangle_nf_subj[:, roi_i, i] = data.reshape(-1)

    export_fig2_no_task(out_xlsx, sheet='Fig5f'+str(task_i), nf_subj=srangle_nf_subj)

out_xlsx

'sourcedata.xlsx'

In [7]:
# Fig5e cwangle summary: first 15 PCs, mean & sem over far-distance samples.
# Source cwangle has no task dimension: nvertex x d x subj x roi.
cwangle_d_subj = np.load(data_path + 'cwangle.npz', allow_pickle=True)['cwangle_d_subj']

nvoxel = cwangle_d_subj.shape[0]
far_i = 1
cwangle_far_subj = np.full((nvoxel, 4 * nsubj, nroi), np.nan)  # PC x 32 samples x roi
for roi_i in range(nroi):
    data = cwangle_d_subj[:, num_nf[far_i][0] : num_nf[far_i][1], :, roi_i]
    cwangle_far_subj[:, :, roi_i] = data.reshape(data.shape[0], -1)

export_pc_summary(out_xlsx, sheet='Fig5e_angle', x=cwangle_far_subj[:15])

out_xlsx


'sourcedata.xlsx'

In [8]:
# Fig5e pcv_i summary: V1 and mFus, first 15 PCs, mean & sem over far-distance samples.
pcvi_d_subj = np.load(data_path + 'cwangle.npz', allow_pickle=True)['pcvi_d_subj']  # nvertex x d x subj x roi x task

nvoxel = pcvi_d_subj.shape[0]
pcvi_nf_subj = np.full((nvoxel, 4 * nsubj, nroi, ntask, len(nf)), np.nan)  # PC x 32 samples x roi x task x nf
for task_i in range(ntask):
    for roi_i in range(nroi):
        for nf_i in range(len(num_nf)):
            data = pcvi_d_subj[:, num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, task_i]
            pcvi_nf_subj[:, :, roi_i, task_i, nf_i] = data.reshape(data.shape[0], -1)

# swap task axis to match figure convention (digit/face)
pcvi_nf_subj[:, :, :, [0, 1], :] = pcvi_nf_subj[:, :, :, [1, 0], :]

far_i = 1
for roi_i in [0, 6]:
    export_pc_summary_by_task(
        out_xlsx,
        sheet=f'Fig5e_pcvi_{roi_labels[roi_i]}',
        x=pcvi_nf_subj[:15, :, roi_i, :, far_i],
        roi_label=roi_labels[roi_i],
    )

out_xlsx


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:277: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sheet = pd.concat(blocks, axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:277: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sheet = pd.concat(blocks, axis=0)


'sourcedata.xlsx'

#### fig 6

In [9]:
# Fig6: far condition, all samples plus mean/sem, split by ROI in one sheet.
deltaIlog_stw_d_subj = np.load(data_path + 'LFI_separate.npz', allow_pickle=True)['deltaIlog_stw_d_subj']

nsample2 = nsubj * np.max(np.diff(num_nf))
ntype_fig6 = deltaIlog_stw_d_subj.shape[-1]
deltaIlog_stw_subj_nf = np.full((nsample2, len(num_nf), nroi, ntype_fig6), np.nan)  # sample x nf x roi x type
for nf_i in range(len(num_nf)):
    for roi_i in range(nroi):
        for type_i in range(ntype_fig6):
            data = deltaIlog_stw_d_subj[num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, type_i].flatten()
            deltaIlog_stw_subj_nf[:len(data), nf_i, roi_i, type_i] = data

far_i = 1
type_labels_fig6 = ['SE', 'DS', 'SR+DW', 'SS+DS+SR+DW(Face Task)']
export_roi_type_samples(
    out_xlsx,
    sheet='Fig6',
    x=deltaIlog_stw_subj_nf[:, far_i, :, :],
    type_labels=type_labels_fig6,
)

out_xlsx


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:308: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sheet = pd.concat(blocks, axis=0)


'sourcedata.xlsx'

#### fig 7

In [8]:
# Fig7: far condition, V2 and mFus, mean/sem only.
nvertex = 100

snanglecos2_d_subj = np.load(data_path + 'snanglecos2.npz', allow_pickle=True)['snanglecos2_d_subj']
snanglecos2_nf_subj = np.full((nvertex, 4 * nsubj, nroi, ntask, len(nf)), np.nan)  # PC x 32 samples x roi x task x nf
for task_i in range(ntask):
    for roi_i in range(nroi):
        for nf_i in range(len(num_nf)):
            data = snanglecos2_d_subj[:, num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, task_i]
            snanglecos2_nf_subj[:, :, roi_i, task_i, nf_i] = data.reshape(data.shape[0], -1)
snanglecos2_nf_subj[:, :, :, [0, 1], :] = snanglecos2_nf_subj[:, :, :, [1, 0], :]

pcvi_d_subj = np.load(data_path + 'cwangle.npz', allow_pickle=True)['pcvi_d_subj']
pcvi_nf_subj = np.full((nvertex, 4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for nf_i in range(len(num_nf)):
            data = pcvi_d_subj[:, num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, task_i]
            pcvi_nf_subj[:, :, roi_i, task_i, nf_i] = data.reshape(data.shape[0], -1)
pcvi_nf_subj[:, :, :, [0, 1], :] = pcvi_nf_subj[:, :, :, [1, 0], :]

sn_d_subj = np.load(data_path + 'sn.npz', allow_pickle=True)['sn_d_subj']
sn_nf_subj = np.full((nvertex, 4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for nf_i in range(len(num_nf)):
            data = sn_d_subj[:, num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, task_i]
            sn_nf_subj[:, :, roi_i, task_i, nf_i] = data.reshape(data.shape[0], -1)
sn_nf_subj[:, :, :, [0, 1], :] = sn_nf_subj[:, :, :, [1, 0], :]

sncum_d_subj = np.load(data_path + 'sn.npz', allow_pickle=True)['sncum_d_subj']
sncum_nf_subj = np.full((nvertex, 4 * nsubj, nroi, ntask, len(nf)), np.nan)
for task_i in range(ntask):
    for roi_i in range(nroi):
        for nf_i in range(len(num_nf)):
            data = sncum_d_subj[:, num_nf[nf_i][0] : num_nf[nf_i][1], :, roi_i, task_i]
            sncum_nf_subj[:, :, roi_i, task_i, nf_i] = data.reshape(data.shape[0], -1)
sncum_nf_subj[:, :, :, [0, 1], :] = sncum_nf_subj[:, :, :, [1, 0], :]

roi_sel = [1, 6]
far_i = 1
data_all = np.full((4 * nsubj, nvertex, ntask, len(roi_sel), 4), np.nan)  # sample x PC x task x selected_roi x metric
for j, roi_i in enumerate(roi_sel):
    data_all[:, :, :, j, 0] = snanglecos2_nf_subj[:, :, roi_i, :, far_i].transpose(1, 0, 2)
    data_all[:, :, :, j, 1] = pcvi_nf_subj[:, :, roi_i, :, far_i].transpose(1, 0, 2)
    data_all[:, :, :, j, 2] = sn_nf_subj[:, :, roi_i, :, far_i].transpose(1, 0, 2)
    data_all[:, :, :, j, 3] = sncum_nf_subj[:, :, roi_i, :, far_i].transpose(1, 0, 2)

metric_labels_fig7 = ['squared_projected_signal', 'relative_variance', 'SNR', 'cumulative_SNR']
export_fig7_summary(out_xlsx, sheet='Fig7', x=data_all, roi_sel=roi_sel, metric_labels=metric_labels_fig7)

out_xlsx


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_46800\2336490431.py:346: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_sheet = pd.concat(blocks, axis=0)


'sourcedata.xlsx'

#### fig s2

In [11]:
export_fig2_like(out_xlsx, sheet='FigS2a', nf_subj=dfnorm_nf_subj)
export_fig2_like(out_xlsx, sheet='FigS2b', nf_subj=v_nf_subj)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df_title, df_out, df_blank], axis=0)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_55068\2336490431.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depr

'sourcedata.xlsx'

#### FigS1-FigS11

In [12]:
# Supplementary source-data export. This writes/replaces only FigS sheets in sourcedata.xlsx.
%run export_supplementary_sourcedata.py
out_xlsx


E:\Rsch-facePRF\Paper\github_public\facePRFattention\plot_figures\export_supplementary_sourcedata.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  _write_sheet(sheet, pd.concat(blocks, axis=0))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1217: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1217: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1217: RuntimeWar

'sourcedata.xlsx'